# 8 — Generation Overview: da fórmula sorteada à fórmula executada

Este notebook reproduz os nove exemplos da seção **4.6 — Generation Overview**
do [relatório em inglês](../../report/english/chapters/implemented_generator/06_generation_overview.tex).
Cada linha parte de uma equação sorteada com configuração e semente explícitas.
Aplicamos as construções do Jacobiano, Shao–Wu com `tanh` nos fatores,
Shao–Wu com `tanh` após o produto e, nos casos lineares, a parametrização de Heaps.
A tabela final mostra a equação efetivamente executada e o propósito de cada exemplo.

As fórmulas foram selecionadas para ilustrar características diferentes. As frequências
de explosão e os efeitos das condições iniciais são avaliados nos experimentos próprios.
O código de geração reutilizado está em `src/tsfmxai/generators/`.

## Como ler este notebook

| Seção | Conteúdo |
|---|---|
| 1–2 | Ambiente, configurações, sementes e escolha dos nove casos. |
| 3 | Sorteio das ASTs originais. |
| 4 | Reescalonamento pelo Jacobiano. |
| 5–6 | As duas posições de `tanh` e seus respectivos critérios. |
| 7 | Construção Heaps para os cinco exemplos lineares. |
| 8 | Verificação numérica das fórmulas e dos fatores usados no relatório. |
| 9 | Tabelas finais: fórmula original, fórmula de cada método e justificativa. |
| 10 | Registro completo do experimento, incluindo configuração, métricas e proveniência. |

Cada equação inclui `epsilon_0(t)`, a inovação adicionada no passo `t`.
Com a mesma equação final, o mesmo histórico e os mesmos valores de inovação,
a trajetória é a mesma, salvo efeitos numéricos.

## 1. Ambiente e convenções

In [1]:
from pathlib import Path
from dataclasses import asdict, replace
import sys
import platform
import hashlib
import subprocess
from datetime import datetime, timezone
import importlib.metadata

ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents)
            if (p / "src/tsfmxai").is_dir())
if str(ROOT / "src") not in sys.path:
    sys.path.insert(0, str(ROOT / "src"))

import numpy as np
import pandas as pd
from IPython.display import display, HTML, JSON
from tsfmxai.generators.random_expression import RandomTreeConfig, sample_random_mechanism
from tsfmxai.generators.contraction import ContractionTreeConfig, build_contractive_mechanism
from tsfmxai.generators.shao_wu import (
    ShaoWuConfig, build_shao_wu_mechanism,
    saturate_product_lags, saturate_product_outputs,
)
from tsfmxai.generators.heaps import build_heaps_var_mechanism
from tsfmxai.generators.trajectory import evaluate_bound_expression
from tsfmxai.representations.expression import to_bound_formula

pd.set_option("display.max_colwidth", None)
pd.set_option("display.max_rows", None)
pd.set_option("display.max_columns", None)
SIGMA = .05
VALIDATION_SEED = 20261027
jacobian_config = ContractionTreeConfig(innovation_bound=np.sqrt(3)*SIGMA)
factor_config = ShaoWuConfig(lag_transform="product_lags", innovation_std=SIGMA)
output_config = ShaoWuConfig(lag_transform="product_output", innovation_std=SIGMA)

noise_policies = pd.DataFrame([
    {"Método": method, "Inovação em cada passo": law, "Desvio-padrão": SIGMA}
    for method, law in [
        ("Original AST", "iid Normal(0, 0.05²)"),
        ("Jacobian", "iid Uniform(-sqrt(3)*0.05, +sqrt(3)*0.05)"),
        ("Shao-Wu: factors", "iid Normal(0, 0.05²)"),
        ("Shao-Wu: output", "iid Normal(0, 0.05²)"),
        ("Heaps", "iid Normal(0, 0.05²)"),
    ]
])
display(noise_policies)

,Método,Inovação em cada passo,Desvio-padrão
0,Original AST,"iid Normal(0, 0.05²)",0.05
1,Jacobian,"iid Uniform(-sqrt(3)*0.05, +sqrt(3)*0.05)",0.05
2,Shao-Wu: factors,"iid Normal(0, 0.05²)",0.05
3,Shao-Wu: output,"iid Normal(0, 0.05²)",0.05
4,Heaps,"iid Normal(0, 0.05²)",0.05


**O que foi feito:** Foram importados os construtores da biblioteca e explicitadas as leis das inovações. O Jacobiano usa ruído limitado para respeitar o domínio verificado. O mesmo desvio-padrão admite leis de ruído diferentes; aqui verificamos equações, sem simular trajetórias.

## 2. Configurações e sementes dos nove exemplos

O número de operadores, a faixa de lags e as probabilidades controlam os sorteios.
Fixamos a magnitude de alguns parâmetros em `0.5` ou `2` para facilitar a leitura.
As sementes foram selecionadas para obter as estruturas desejadas; a tabela representa
uma coleção didática de casos, e cada caso permanece reproduzível.

Em `cases`, as funções `lambda` descrevem as fórmulas esperadas para a verificação.
Os exemplos efetivos serão sorteados por `sample_random_mechanism` na próxima seção.
Nessas funções, `z[0]` significa o lag 1, `z[1]` o lag 2, e assim por diante.

In [2]:
base = RandomTreeConfig(minimum_operators=0, maximum_operators=0,
        time_probability=0., parameter_probability=0., lag_probability=1.,
        minimum_lag=1, maximum_lag=1, minimum_parameter_magnitude=.5,
        maximum_parameter_magnitude=.5, add_probability=.65,
        coefficient_gate_lags=False)

cases = [
        ("R1", base, 0, lambda z:z[0], lambda z:np.tanh(z[0]), [1.], 0.,
         (.855, .76, .76)),
        ("R2", replace(base, minimum_operators=1, maximum_operators=1,
         parameter_probability=1., add_probability=0., minimum_parameter_magnitude=2.,
         maximum_parameter_magnitude=2.), 0, lambda z:2*z[0], None, [2.], 0.,
         (.4275, .38, .38)),
        ("R3", replace(base, minimum_operators=1, maximum_operators=1,
         parameter_probability=1., add_probability=1.), 2, lambda z:z[0]-.5,
         None, [1.], -.5, (.95*(1-np.sqrt(3)*.05)/1.5, .76, .76)),
        ("R4", replace(base, minimum_operators=1, maximum_operators=1,
         maximum_lag=3, add_probability=1.), 0, lambda z:z[0]+z[2], None,
         [1.,0.,1.], 0., (.855/(1+1.25**2), .38, .38)),
        ("R5", replace(base, minimum_lag=12, maximum_lag=12), 0,
         lambda z:z[11], None, [0.]*11+[1.], 0., (.855/1.25**11, .76, .76)),
        ("R6", replace(base, minimum_operators=1, maximum_operators=1,
         maximum_lag=2, add_probability=0.), 0, lambda z:z[0]*z[1],
         lambda z:np.tanh(z[0])*np.tanh(z[1]), None, 0., (.38,.38,.95)),
        ("R7", replace(base, minimum_operators=1, maximum_operators=1,
         add_probability=0.), 0, lambda z:z[0]**2, lambda z:np.tanh(z[0])**2,
         None, 0., (.4275,.38,.95)),
        ("R8", replace(base, minimum_operators=3, maximum_operators=3,
         maximum_lag=3, add_probability=0.), 5, lambda z:z[0]**2*z[1]*z[2],
         lambda z:np.tanh(z[0])**2*np.tanh(z[1])*np.tanh(z[2]), None, 0.,
         (.855/(2+1.25+1.25**2), .19, .95)),
        ("R9", replace(base, minimum_operators=3, maximum_operators=3,
         maximum_lag=2, parameter_probability=.4, add_probability=.5), 1438,
         lambda z:.5*z[0]+z[0]*z[1],
         lambda z:.5*z[0]+np.tanh(z[0])*np.tanh(z[1]), None, 0.,
         (.855/(1.5+1.25), .304, .95)),
    ]

**O que foi feito:** Foram declaradas as mesmas configurações e sementes do verificador do relatório. Cada tupla guarda o identificador, a configuração, a semente, as funções de referência, a entrada linear opcional para Heaps, o intercepto e os três multiplicadores esperados.

In [3]:
WHY = {
    "R1": "Realimentação unitária: o original é um passeio aleatório; os tratamentos mudam a memória.",
    "R2": "Realimentação amplificadora: J/F/P coincidem com R1 após a escala; Heaps mantém uma diferença.",
    "R3": "Intercepto: a escala da AST altera também o deslocamento; Heaps preserva-o pela convenção declarada.",
    "R4": "Lags separados: pesos do Jacobiano e coeficiente intermediário introduzido pelo Heaps.",
    "R5": "Lag 12: memória distante aumenta a atenuação exigida pelo Jacobiano.",
    "R6": "Produto de dois lags distintos: comparar polinômio, fatores saturados e produto saturado.",
    "R7": "Lag repetido: duas folhas representam o quadrado da mesma coordenada.",
    "R8": "Quatro fatores: maior orçamento de sensibilidade; saturação do produto completo uma única vez.",
    "R9": "Termo linear mais interação: tanh seletiva preserva o termo linear antes da escala global.",
}
display(pd.DataFrame([
    {"Caso": name, "Semente": seed, "Operadores": cfg.minimum_operators,
     "Lags permitidos": f"{cfg.minimum_lag} a {cfg.maximum_lag}",
     "Por que este exemplo?": WHY[name]}
    for name, cfg, seed, *_ in cases
]))

,Caso,Semente,Operadores,Lags permitidos,Por que este exemplo?
0,R1,0,0,1 a 1,Realimentação unitária: o original é um passeio aleatório; os tratamentos mudam a memória.
1,R2,0,1,1 a 1,Realimentação amplificadora: J/F/P coincidem com R1 após a escala; Heaps mantém uma diferença.
2,R3,2,1,1 a 1,Intercepto: a escala da AST altera também o deslocamento; Heaps preserva-o pela convenção declarada.
3,R4,0,1,1 a 3,Lags separados: pesos do Jacobiano e coeficiente intermediário introduzido pelo Heaps.
4,R5,0,0,12 a 12,Lag 12: memória distante aumenta a atenuação exigida pelo Jacobiano.
5,R6,0,1,1 a 2,"Produto de dois lags distintos: comparar polinômio, fatores saturados e produto saturado."
6,R7,0,1,1 a 1,Lag repetido: duas folhas representam o quadrado da mesma coordenada.
7,R8,5,3,1 a 3,Quatro fatores: maior orçamento de sensibilidade; saturação do produto completo uma única vez.
8,R9,1438,3,1 a 2,Termo linear mais interação: tanh seletiva preserva o termo linear antes da escala global.


**O que foi feito:** A tabela relaciona cada semente e configuração à característica que queremos observar. A mesma semente pode gerar fórmulas diferentes quando a configuração muda.

## 3. Original AST: sortear uma fórmula por linha

In [4]:
originals = {
    name: sample_random_mechanism(cfg, seed=seed)
    for name, cfg, seed, *_ in cases
}
formula_rows = {
    name: {"Caso": name, "Semente": seed,
           "Original AST": f"x_0(t) = {originals[name].formula} + epsilon_0(t)"}
    for name, cfg, seed, *_ in cases
}
display(pd.DataFrame(formula_rows.values()))

,Caso,Semente,Original AST
0,R1,0,x_0(t) = x_0(t-1) + epsilon_0(t)
1,R2,0,x_0(t) = (x_0(t-1) * 2) + epsilon_0(t)
2,R3,2,x_0(t) = (x_0(t-1) + -0.5) + epsilon_0(t)
3,R4,0,x_0(t) = (x_0(t-1) + x_0(t-3)) + epsilon_0(t)
4,R5,0,x_0(t) = x_0(t-12) + epsilon_0(t)
5,R6,0,x_0(t) = (x_0(t-1) * x_0(t-2)) + epsilon_0(t)
6,R7,0,x_0(t) = (x_0(t-1) * x_0(t-1)) + epsilon_0(t)
7,R8,5,x_0(t) = (x_0(t-1) * (x_0(t-2) * (x_0(t-3) * x_0(t-1)))) + epsilon_0(t)
8,R9,1438,x_0(t) = ((x_0(t-1) * x_0(t-2)) + (0.5 * x_0(t-1))) + epsilon_0(t)


**O que foi feito:** Cada AST foi realmente sorteada pelo código do projeto. A equação original mantém o resultado do sorteio e recebe a inovação aditiva em cada passo. Os objetos ficam guardados para que os métodos seguintes partam da mesma AST.

## 4. Jacobiano: escalar a expressão inteira

O construtor calcula limites de amplitude e de sensibilidade no domínio
$[-1,1]^L$, incluindo a atualização da memória. Usa crescimento dos pesos `1.25`,
ganho-alvo `0.9` e multiplicador de segurança `0.95`.
O fator final multiplica toda a expressão determinística; a inovação é adicionada depois.

In [5]:
jacobians = {}
for name, sample in originals.items():
    jacobians[name] = build_contractive_mechanism(
        sample.expression, sample.binding, jacobian_config
    )
    formula_rows[name]["Jacobian"] = jacobians[name].formula

display(pd.DataFrame([
    {"Caso": name, "Multiplicador": mechanism.scale,
     "Equação obtida": mechanism.formula,
     "Verificação aprovada": mechanism.certificate.holds}
    for name, mechanism in jacobians.items()
]))

,Caso,Multiplicador,Equação obtida,Verificação aprovada
0,R1,0.855000,x_0(t) = (0.855 * x_0(t-1)) + epsilon_0(t),True
1,R2,0.427500,x_0(t) = (0.4275 * (x_0(t-1) * 2)) + epsilon_0(t),True
2,R3,0.578485,x_0(t) = (0.578485 * (x_0(t-1) + -0.5)) + epsilon_0(t),True
3,R4,0.333659,x_0(t) = (0.333659 * (x_0(t-1) + x_0(t-3))) + epsilon_0(t),True
4,R5,0.073444,x_0(t) = (0.0734439 * x_0(t-12)) + epsilon_0(t),True
5,R6,0.380000,x_0(t) = (0.38 * (x_0(t-1) * x_0(t-2))) + epsilon_0(t),True
6,R7,0.427500,x_0(t) = (0.4275 * (x_0(t-1) * x_0(t-1))) + epsilon_0(t),True
7,R8,0.177662,x_0(t) = (0.177662 * (x_0(t-1) * (x_0(t-2) * (x_0(t-3) * x_0(t-1))))) + epsilon_0(t),True
8,R9,0.310909,x_0(t) = (0.310909 * ((x_0(t-1) * x_0(t-2)) + (0.5 * x_0(t-1)))) + epsilon_0(t),True


**O que foi feito:** O construtor aplicou o reescalonamento e verificou contração e contenção no domínio. O lag distante de R5 recebe uma atenuação maior. A tabela mostra o multiplicador explicitamente, incluindo seu efeito sobre o intercepto de R3.

## 5. Shao–Wu: `tanh` nos fatores

Primeiro transformamos as folhas de lag que participam de produtos não lineares.
Depois calculamos a escala global. Por exemplo,
$x_{t-1}x_{t-2}$ torna-se $\tanh(x_{t-1})\tanh(x_{t-2})$.
O modo verifica sensibilidade global e a condição de momento de referência,
conforme a especialização do Teorema 5.1 apresentada na seção 5.5 do relatório.

In [6]:
factors = {}
for name, sample in originals.items():
    transformed = saturate_product_lags(sample.expression)
    factors[name] = build_shao_wu_mechanism(
        (transformed,), (sample.binding,), factor_config
    )
    formula_rows[name]["Shao-Wu: factors"] = factors[name].formulas[0]

display(pd.DataFrame([
    {"Caso": name, "Multiplicador": mechanism.scale,
     "Equação obtida": mechanism.formulas[0],
     "Propriedade verificada": mechanism.certificate.guarantee}
    for name, mechanism in factors.items()
]))

,Caso,Multiplicador,Equação obtida,Propriedade verificada
0,R1,0.760,x_0(t) = (0.76 * x_0(t-1)) + epsilon_0(t),geometric_moment_contraction
1,R2,0.380,x_0(t) = (0.38 * (x_0(t-1) * 2)) + epsilon_0(t),geometric_moment_contraction
2,R3,0.760,x_0(t) = (0.76 * (x_0(t-1) + -0.5)) + epsilon_0(t),geometric_moment_contraction
3,R4,0.380,x_0(t) = (0.38 * (x_0(t-1) + x_0(t-3))) + epsilon_0(t),geometric_moment_contraction
4,R5,0.760,x_0(t) = (0.76 * x_0(t-12)) + epsilon_0(t),geometric_moment_contraction
5,R6,0.380,x_0(t) = (0.38 * (tanh(x_0(t-1)) * tanh(x_0(t-2)))) + epsilon_0(t),geometric_moment_contraction
6,R7,0.380,x_0(t) = (0.38 * (tanh(x_0(t-1)) * tanh(x_0(t-1)))) + epsilon_0(t),geometric_moment_contraction
7,R8,0.190,x_0(t) = (0.19 * (tanh(x_0(t-1)) * (tanh(x_0(t-2)) * (tanh(x_0(t-3)) * tanh(x_0(t-1)))))) + epsilon_0(t),geometric_moment_contraction
8,R9,0.304,x_0(t) = (0.304 * ((tanh(x_0(t-1)) * tanh(x_0(t-2))) + (0.5 * x_0(t-1)))) + epsilon_0(t),geometric_moment_contraction


**O que foi feito:** A tanh foi aplicada aos fatores dos produtos não lineares antes da escala. Termos lineares isolados permanecem lineares. A propriedade declarada pelo objeto é geometric_moment_contraction.

## 6. Shao–Wu: `tanh` após o produto

Neste modo, $x_{t-1}x_{t-2}$ torna-se $\tanh(x_{t-1}x_{t-2})$.
O produto conectado inteiro recebe uma saturação, inclusive quando tem quatro fatores.
A verificação usada é a envoltória de segundo momento desenvolvida no projeto.
O limite de amplitude e o limite de sensibilidade são propriedades distintas.

In [7]:
outputs = {}
for name, sample in originals.items():
    transformed = saturate_product_outputs(sample.expression)
    outputs[name] = build_shao_wu_mechanism(
        (transformed,), (sample.binding,), output_config
    )
    formula_rows[name]["Shao-Wu: output"] = outputs[name].formulas[0]

display(pd.DataFrame([
    {"Caso": name, "Multiplicador": mechanism.scale,
     "Equação obtida": mechanism.formulas[0],
     "Propriedade verificada": mechanism.certificate.guarantee}
    for name, mechanism in outputs.items()
]))

,Caso,Multiplicador,Equação obtida,Propriedade verificada
0,R1,0.76,x_0(t) = (0.76 * x_0(t-1)) + epsilon_0(t),uniform_second_moment_bound
1,R2,0.38,x_0(t) = (0.38 * (x_0(t-1) * 2)) + epsilon_0(t),uniform_second_moment_bound
2,R3,0.76,x_0(t) = (0.76 * (x_0(t-1) + -0.5)) + epsilon_0(t),uniform_second_moment_bound
3,R4,0.38,x_0(t) = (0.38 * (x_0(t-1) + x_0(t-3))) + epsilon_0(t),uniform_second_moment_bound
4,R5,0.76,x_0(t) = (0.76 * x_0(t-12)) + epsilon_0(t),uniform_second_moment_bound
5,R6,0.95,x_0(t) = (0.95 * tanh((x_0(t-1) * x_0(t-2)))) + epsilon_0(t),uniform_second_moment_bound
6,R7,0.95,x_0(t) = (0.95 * tanh((x_0(t-1) * x_0(t-1)))) + epsilon_0(t),uniform_second_moment_bound
7,R8,0.95,x_0(t) = (0.95 * tanh((((x_0(t-1) * x_0(t-2)) * x_0(t-3)) * x_0(t-1)))) + epsilon_0(t),uniform_second_moment_bound
8,R9,0.95,x_0(t) = (0.95 * (tanh((x_0(t-1) * x_0(t-2))) + (0.5 * x_0(t-1)))) + epsilon_0(t),uniform_second_moment_bound


**O que foi feito:** Foi saturado o resultado de cada produto não linear, mantendo os coeficientes externos. A verificação retorna uniform_second_moment_bound. Nos produtos puros, a escala fica em 0.95; o fator de segurança ainda é aplicado mesmo quando o crescimento já está limitado.

## 7. Heaps: construção linear com parâmetros correspondentes

O construtor Heaps recebe matrizes livres $U_\ell$ e reconstrói os coeficientes
de um VAR estacionário. Para comparar com uma fórmula linear sorteada, usamos
seus coeficientes como entradas numéricas de $U_\ell$, preenchendo lags ausentes com zero.
Essa correspondência é uma convenção didática explícita: parâmetros livres e
coeficientes AR finais pertencem a espaços diferentes.

Após reconstruir os coeficientes, escolhemos a média estacionária para preservar
o intercepto original: `intercept / (1 - coefficients.sum())`.
Nos casos não lineares, a tabela marca o limite do vocabulário VAR.

In [8]:
heaps = {}
heaps_inputs = []
for name, cfg, seed, expected_original, expected_factor, free, intercept, scales in cases:
    if free is None:
        formula_rows[name]["Heaps"] = "Fora do vocabulário VAR linear"
        continue
    u = np.asarray(free)[:, None, None]
    covariance = np.array([[SIGMA**2]])
    provisional = build_heaps_var_mechanism(u, covariance)
    coefficients = np.asarray(provisional.coefficients)[:, 0, 0]
    stationary_mean = intercept / (1 - coefficients.sum())
    heaps[name] = build_heaps_var_mechanism(
        u, covariance, stationary_mean=np.array([stationary_mean])
    )
    formula_rows[name]["Heaps"] = heaps[name].formulas[0]
    heaps_inputs.append({"Caso": name, "Entradas livres U": free,
                        "Coeficientes AR finais": coefficients.tolist(),
                        "Média estacionária": stationary_mean,
                        "Equação obtida": heaps[name].formulas[0]})

display(pd.DataFrame(heaps_inputs))

,Caso,Entradas livres U,Coeficientes AR finais,Média estacionária,Equação obtida
0,R1,[1.0],[0.7071067811865475],0.000000,x_0(t) = (0 + (0.707107 * x_0(t-1))) + epsilon_0(t)
1,R2,[2.0],[0.8944271909999159],0.000000,x_0(t) = (0 + (0.894427 * x_0(t-1))) + epsilon_0(t)
2,R3,[1.0],[0.7071067811865475],-1.707107,x_0(t) = (-0.5 + (0.707107 * x_0(t-1))) + epsilon_0(t)
3,R4,"[1.0, 0.0, 1.0]","[0.7071067811865475, -0.4999999999999999, 0.7071067811865475]",0.000000,x_0(t) = (((0 + (0.707107 * x_0(t-1))) + (-0.5 * x_0(t-2))) + (0.707107 * x_0(t-3))) + epsilon_0(t)
4,R5,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.0]","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.7071067811865475]",0.000000,x_0(t) = (0 + (0.707107 * x_0(t-12))) + epsilon_0(t)


**O que foi feito:** A reconstrução foi feita conjuntamente para todos os lags. Em R4, entradas livres não nulas nos lags 1 e 3 produzem também um coeficiente no lag 2. As cinco equações lineares foram obtidas pelo construtor real; as quatro não lineares ficam explicitamente fora desse escopo.

## 8. Conferir os cálculos e as fórmulas do relatório

Comparamos a execução de cada AST com a expressão matemática esperada em 32 históricos
determinísticos, sorteados pela semente de validação. A avaliação pontual abaixo usa
o termo determinístico; a inovação aditiva comum cancelaria na diferença.
Também conferimos os multiplicadores, os coeficientes de Heaps e os registros de verificação.
Os históricos de teste são entradas pontuais para verificar identidade de fórmulas.

In [9]:
validation_histories = np.random.default_rng(VALIDATION_SEED).uniform(-1, 1, (32, 12))
expected_heaps = {
    "R1": [1/np.sqrt(2)], "R2": [2/np.sqrt(5)], "R3": [1/np.sqrt(2)],
    "R4": [1/np.sqrt(2), -.5, 1/np.sqrt(2)], "R5": [0.]*11 + [1/np.sqrt(2)],
}
validation_rows = []
for name, cfg, seed, expected_original, expected_factor, free, intercept, scales in cases:
    sample, j, f, o = originals[name], jacobians[name], factors[name], outputs[name]
    np.testing.assert_allclose([j.scale, f.scale, o.scale], scales, rtol=1e-13)
    assert j.certificate.holds and f.certificate.holds and o.certificate.holds
    errors = {method: [] for method in ["Original AST", "Jacobian", "Shao-Wu: factors", "Shao-Wu: output"]}
    if name in heaps:
        errors["Heaps"] = []
        coefficients = np.asarray(heaps[name].coefficients)[:, 0, 0]
        np.testing.assert_allclose(coefficients, expected_heaps[name], atol=1e-12)
        assert heaps[name].certificate.holds
    for z in validation_histories:
        # Library histories are oldest-first; z is newest-first.
        evaluate = lambda e, b: evaluate_bound_expression(e, b, time=12, histories=z[::-1][None, :])
        expected_f = expected_original(z) if free is not None else expected_factor(z)
        expected_o = expected_original(z) if free is not None else (
            .5*z[0] + np.tanh(z[0]*z[1]) if name == "R9" else np.tanh(expected_original(z)))
        comparisons = [
            ("Original AST", evaluate(sample.expression, sample.binding), expected_original(z)),
            ("Jacobian", evaluate(j.expression, j.binding), scales[0]*expected_original(z)),
            ("Shao-Wu: factors", evaluate(f.expressions[0], f.bindings[0]), scales[1]*expected_f),
            ("Shao-Wu: output", evaluate(o.expressions[0], o.bindings[0]), scales[2]*expected_o),
        ]
        if name in heaps:
            h = heaps[name]
            comparisons.append(("Heaps", evaluate(h.expressions[0], h.bindings[0]),
                                intercept + coefficients @ z[:len(coefficients)]))
        for method, actual, expected in comparisons:
            error = abs(actual - expected)
            assert error <= 1e-12, (name, method, error)
            errors[method].append(error)
    for method, values in errors.items():
        validation_rows.append({"Caso": name, "Método": method, "Históricos testados": len(values),
                                "Máximo erro absoluto": max(values), "Aprovado": True})
validation = pd.DataFrame(validation_rows)
display(validation)
assert len(validation) == 41
print(f"41 combinações caso/método; {validation['Históricos testados'].sum()} avaliações pontuais aprovadas.")

,Caso,Método,Históricos testados,Máximo erro absoluto,Aprovado
0,R1,Original AST,32,0.000000e+00,True
1,R1,Jacobian,32,0.000000e+00,True
2,R1,Shao-Wu: factors,32,0.000000e+00,True
3,R1,Shao-Wu: output,32,0.000000e+00,True
4,R1,Heaps,32,0.000000e+00,True
5,R2,Original AST,32,0.000000e+00,True
6,R2,Jacobian,32,0.000000e+00,True
7,R2,Shao-Wu: factors,32,0.000000e+00,True
8,R2,Shao-Wu: output,32,0.000000e+00,True
9,R2,Heaps,32,0.000000e+00,True


41 combinações caso/método; 1312 avaliações pontuais aprovadas.


**O que foi feito:** Foram verificadas 41 combinações: nove originais, 27 transformações de AST e cinco construções Heaps. Cada combinação foi avaliada em 32 históricos, totalizando 1.312 verificações pontuais. As fórmulas esperadas são independentes da avaliação da AST, e os multiplicadores são comparados com os cálculos do relatório.

## 9. Tabelas finais — entrada, alteração por método e característica ilustrada

In [10]:
for name in formula_rows:
    formula_rows[name]["Por que este exemplo?"] = WHY[name]
comparison = pd.DataFrame(formula_rows.values()).set_index("Caso")

# This explanatory row is displayed separately from the sampled mechanisms.
operations = pd.DataFrame([{
    "Semente": "Definida em cada exemplo abaixo.",
    "Original AST": "Mantém a fórmula sorteada e adiciona uma inovação em cada passo.",
    "Jacobian": "Multiplica toda a parte determinística por uma escala calculada para atender aos limites de contração e de domínio. Em R2: 0.4275 × (2x) = 0.855x. A inovação é adicionada depois.",
    "Shao-Wu: factors": "Aplica tanh aos lags que participam de produtos não lineares; depois multiplica toda a expressão por uma escala que limita a sensibilidade ao passado. Em R2, o termo linear recebe apenas a escala: 0.38 × (2x) = 0.76x. Adiciona a inovação depois.",
    "Shao-Wu: output": "Calcula cada produto não linear completo e aplica tanh ao resultado; depois reescala toda a expressão para verificar o limite de segundo momento. Em R6: 0.95 × tanh(x(t−1)x(t−2)). Adiciona a inovação depois.",
    "Heaps": "Usa os coeficientes lineares originais como parâmetros livres U, pela convenção desta comparação, e reconstrói conjuntamente os coeficientes AR estáveis. Para AR(1), a = u/sqrt(1+u²): u=2 produz a≈0.894427. Preserva o intercepto pela média escolhida e adiciona a inovação.",
    "Por que este exemplo?": "Esta linha explica a operação geral; cada linha seguinte mostra sua aplicação a uma fórmula sorteada.",
}], index=pd.Index(["Operação"], name="Caso"))
for title, block in [("Casos lineares e afins — R1 a R5", comparison.iloc[:5]),
                     ("Casos não lineares — R6 a R9", comparison.iloc[5:])]:
    print(title)
    # Styling only: formulas are read from the constructed mechanism objects.
    display_block = pd.concat([operations, block])
    styled = display_block.style.set_properties(**{
        "white-space": "normal", "min-width": "190px", "max-width": "270px",
        "text-align": "left", "vertical-align": "top", "padding": "8px",
    })
    styled = styled.set_properties(subset=pd.IndexSlice[["Operação"], :], **{
        "background-color": "#edf3f8", "font-weight": "bold",
    })
    display(HTML('<div style="overflow-x:auto;max-width:100%">' + styled.to_html() + '</div>'))

display(pd.DataFrame([
    {"Caso": name, "Escala Jacobian": jacobians[name].scale,
     "Escala Shao-Wu factors": factors[name].scale,
     "Escala Shao-Wu output": outputs[name].scale}
    for name in originals
]))

Casos lineares e afins — R1 a R5


,Semente,Original AST,Jacobian,Shao-Wu: factors,Shao-Wu: output,Heaps,Por que este exemplo?
Caso,,,,,,,
Operação,Definida em cada exemplo abaixo.,Mantém a fórmula sorteada e adiciona uma inovação em cada passo.,Multiplica toda a parte determinística por uma escala calculada para atender aos limites de contração e de domínio. Em R2: 0.4275 × (2x) = 0.855x. A inovação é adicionada depois.,"Aplica tanh aos lags que participam de produtos não lineares; depois multiplica toda a expressão por uma escala que limita a sensibilidade ao passado. Em R2, o termo linear recebe apenas a escala: 0.38 × (2x) = 0.76x. Adiciona a inovação depois.",Calcula cada produto não linear completo e aplica tanh ao resultado; depois reescala toda a expressão para verificar o limite de segundo momento. Em R6: 0.95 × tanh(x(t−1)x(t−2)). Adiciona a inovação depois.,"Usa os coeficientes lineares originais como parâmetros livres U, pela convenção desta comparação, e reconstrói conjuntamente os coeficientes AR estáveis. Para AR(1), a = u/sqrt(1+u²): u=2 produz a≈0.894427. Preserva o intercepto pela média escolhida e adiciona a inovação.",Esta linha explica a operação geral; cada linha seguinte mostra sua aplicação a uma fórmula sorteada.
R1,0,x_0(t) = x_0(t-1) + epsilon_0(t),x_0(t) = (0.855 * x_0(t-1)) + epsilon_0(t),x_0(t) = (0.76 * x_0(t-1)) + epsilon_0(t),x_0(t) = (0.76 * x_0(t-1)) + epsilon_0(t),x_0(t) = (0 + (0.707107 * x_0(t-1))) + epsilon_0(t),Realimentação unitária: o original é um passeio aleatório; os tratamentos mudam a memória.
R2,0,x_0(t) = (x_0(t-1) * 2) + epsilon_0(t),x_0(t) = (0.4275 * (x_0(t-1) * 2)) + epsilon_0(t),x_0(t) = (0.38 * (x_0(t-1) * 2)) + epsilon_0(t),x_0(t) = (0.38 * (x_0(t-1) * 2)) + epsilon_0(t),x_0(t) = (0 + (0.894427 * x_0(t-1))) + epsilon_0(t),Realimentação amplificadora: J/F/P coincidem com R1 após a escala; Heaps mantém uma diferença.
R3,2,x_0(t) = (x_0(t-1) + -0.5) + epsilon_0(t),x_0(t) = (0.578485 * (x_0(t-1) + -0.5)) + epsilon_0(t),x_0(t) = (0.76 * (x_0(t-1) + -0.5)) + epsilon_0(t),x_0(t) = (0.76 * (x_0(t-1) + -0.5)) + epsilon_0(t),x_0(t) = (-0.5 + (0.707107 * x_0(t-1))) + epsilon_0(t),Intercepto: a escala da AST altera também o deslocamento; Heaps preserva-o pela convenção declarada.
R4,0,x_0(t) = (x_0(t-1) + x_0(t-3)) + epsilon_0(t),x_0(t) = (0.333659 * (x_0(t-1) + x_0(t-3))) + epsilon_0(t),x_0(t) = (0.38 * (x_0(t-1) + x_0(t-3))) + epsilon_0(t),x_0(t) = (0.38 * (x_0(t-1) + x_0(t-3))) + epsilon_0(t),x_0(t) = (((0 + (0.707107 * x_0(t-1))) + (-0.5 * x_0(t-2))) + (0.707107 * x_0(t-3))) + epsilon_0(t),Lags separados: pesos do Jacobiano e coeficiente intermediário introduzido pelo Heaps.
R5,0,x_0(t) = x_0(t-12) + epsilon_0(t),x_0(t) = (0.0734439 * x_0(t-12)) + epsilon_0(t),x_0(t) = (0.76 * x_0(t-12)) + epsilon_0(t),x_0(t) = (0.76 * x_0(t-12)) + epsilon_0(t),x_0(t) = (0 + (0.707107 * x_0(t-12))) + epsilon_0(t),Lag 12: memória distante aumenta a atenuação exigida pelo Jacobiano.


Casos não lineares — R6 a R9


,Semente,Original AST,Jacobian,Shao-Wu: factors,Shao-Wu: output,Heaps,Por que este exemplo?
Caso,,,,,,,
Operação,Definida em cada exemplo abaixo.,Mantém a fórmula sorteada e adiciona uma inovação em cada passo.,Multiplica toda a parte determinística por uma escala calculada para atender aos limites de contração e de domínio. Em R2: 0.4275 × (2x) = 0.855x. A inovação é adicionada depois.,"Aplica tanh aos lags que participam de produtos não lineares; depois multiplica toda a expressão por uma escala que limita a sensibilidade ao passado. Em R2, o termo linear recebe apenas a escala: 0.38 × (2x) = 0.76x. Adiciona a inovação depois.",Calcula cada produto não linear completo e aplica tanh ao resultado; depois reescala toda a expressão para verificar o limite de segundo momento. Em R6: 0.95 × tanh(x(t−1)x(t−2)). Adiciona a inovação depois.,"Usa os coeficientes lineares originais como parâmetros livres U, pela convenção desta comparação, e reconstrói conjuntamente os coeficientes AR estáveis. Para AR(1), a = u/sqrt(1+u²): u=2 produz a≈0.894427. Preserva o intercepto pela média escolhida e adiciona a inovação.",Esta linha explica a operação geral; cada linha seguinte mostra sua aplicação a uma fórmula sorteada.
R6,0,x_0(t) = (x_0(t-1) * x_0(t-2)) + epsilon_0(t),x_0(t) = (0.38 * (x_0(t-1) * x_0(t-2))) + epsilon_0(t),x_0(t) = (0.38 * (tanh(x_0(t-1)) * tanh(x_0(t-2)))) + epsilon_0(t),x_0(t) = (0.95 * tanh((x_0(t-1) * x_0(t-2)))) + epsilon_0(t),Fora do vocabulário VAR linear,"Produto de dois lags distintos: comparar polinômio, fatores saturados e produto saturado."
R7,0,x_0(t) = (x_0(t-1) * x_0(t-1)) + epsilon_0(t),x_0(t) = (0.4275 * (x_0(t-1) * x_0(t-1))) + epsilon_0(t),x_0(t) = (0.38 * (tanh(x_0(t-1)) * tanh(x_0(t-1)))) + epsilon_0(t),x_0(t) = (0.95 * tanh((x_0(t-1) * x_0(t-1)))) + epsilon_0(t),Fora do vocabulário VAR linear,Lag repetido: duas folhas representam o quadrado da mesma coordenada.
R8,5,x_0(t) = (x_0(t-1) * (x_0(t-2) * (x_0(t-3) * x_0(t-1)))) + epsilon_0(t),x_0(t) = (0.177662 * (x_0(t-1) * (x_0(t-2) * (x_0(t-3) * x_0(t-1))))) + epsilon_0(t),x_0(t) = (0.19 * (tanh(x_0(t-1)) * (tanh(x_0(t-2)) * (tanh(x_0(t-3)) * tanh(x_0(t-1)))))) + epsilon_0(t),x_0(t) = (0.95 * tanh((((x_0(t-1) * x_0(t-2)) * x_0(t-3)) * x_0(t-1)))) + epsilon_0(t),Fora do vocabulário VAR linear,Quatro fatores: maior orçamento de sensibilidade; saturação do produto completo uma única vez.
R9,1438,x_0(t) = ((x_0(t-1) * x_0(t-2)) + (0.5 * x_0(t-1))) + epsilon_0(t),x_0(t) = (0.310909 * ((x_0(t-1) * x_0(t-2)) + (0.5 * x_0(t-1)))) + epsilon_0(t),x_0(t) = (0.304 * ((tanh(x_0(t-1)) * tanh(x_0(t-2))) + (0.5 * x_0(t-1)))) + epsilon_0(t),x_0(t) = (0.95 * (tanh((x_0(t-1) * x_0(t-2))) + (0.5 * x_0(t-1)))) + epsilon_0(t),Fora do vocabulário VAR linear,Termo linear mais interação: tanh seletiva preserva o termo linear antes da escala global.


,Caso,Escala Jacobian,Escala Shao-Wu factors,Escala Shao-Wu output
0,R1,0.855000,0.760,0.76
1,R2,0.427500,0.380,0.38
2,R3,0.578485,0.760,0.76
3,R4,0.333659,0.380,0.38
4,R5,0.073444,0.760,0.76
5,R6,0.380000,0.380,0.95
6,R7,0.427500,0.380,0.95
7,R8,0.177662,0.190,0.95
8,R9,0.310909,0.304,0.95


**O que foi feito:** As tabelas foram montadas a partir dos objetos efetivamente construídos, mantendo parênteses e multiplicadores para deixar a transformação visível. A apresentação é algebricamente equivalente às fórmulas simplificadas do relatório. As inovações aparecem em todas as equações executáveis; suas leis estão na seção 1. A primeira linha, destacada como Operação, explica o que cada método faz antes de apresentar os exemplos.

### Leitura dos resultados

- R1 e R2 mostram que fórmulas originais diferentes podem chegar à mesma equação após reescalonamento.
- R3 mostra que a escala global também altera o intercepto.
- R4 e R5 mostram o papel da posição dos lags; o Heaps pode alterar o suporte dos coeficientes.
- R6–R8 isolam produto entre lags, repetição de um lag e aumento do grau do produto.
- R9 mostra que a saturação seletiva preserva o termo linear, enquanto a escala global afeta ambos os termos.

As propriedades verificadas dependem da família: domínio invariante e contração no Jacobiano,
contração em momento no modo dos fatores, limite de segundo momento no modo do produto,
e coeficientes VAR estáveis no Heaps. Uma comparação de trajetórias também precisa declarar
as condições iniciais e o compartilhamento das inovações.

### Referências e código

- Shao e Wu (2007), *Asymptotic Spectral Theory for Nonlinear Time Series*,
  Teorema 5.1, pp. 11–12 da versão eletrônica: [DOI](https://doi.org/10.1214/009053606000001479).
- Heaps (2023), *Enforcing Stationarity through the Prior in Vector Autoregressions*,
  seções 2.1–2.2 e Apêndice: [DOI](https://doi.org/10.1080/10618600.2022.2079648).
- [Discussão do Jacobiano no relatório](../../report/english/chapters/stability_control/04_domain_contraction.tex),
  incluindo as referências a Tran et al. e Laraba et al.
- [Verificador em linha de comando](../../scripts/check_generation_overview.py).

As referências do notebook usam caminhos relativos à sua pasta.

## 10. Registro completo do experimento

In [11]:
source_paths = [
    "src/tsfmxai/generators/random_expression.py", "src/tsfmxai/generators/contraction.py",
    "src/tsfmxai/generators/shao_wu.py", "src/tsfmxai/generators/heaps.py",
    "src/tsfmxai/generators/trajectory.py", "src/tsfmxai/representations/expression.py",
    "scripts/check_generation_overview.py",
    "report/english/chapters/implemented_generator/06_generation_overview.tex",
]
git_revision = subprocess.run(["git", "rev-parse", "HEAD"], cwd=ROOT, capture_output=True, text=True, check=True).stdout.strip()
git_status = subprocess.run(["git", "status", "--porcelain", "--", *source_paths], cwd=ROOT,
                            capture_output=True, text=True, check=True).stdout.splitlines()
record = {
    "research_question": "Como uma mesma AST sorteada muda sob cada construção de controle de estabilidade?",
    "hypothesis": "As regras alteram coeficientes, saturações e eventualmente suporte de lags; as fórmulas executadas devem corresponder aos cálculos declarados.",
    "selection": "Nove casos ilustrativos selecionados por configuração e semente; magnitudes dos parâmetros fixadas para legibilidade.",
    "configuration": {
        "cases": [{"id": name, "seed": seed, "sampler": asdict(cfg),
                   "heaps_free_entries": free, "heaps_preserved_intercept": intercept,
                   "expected_scales": list(scales)}
                  for name, cfg, seed, original, factor, free, intercept, scales in cases],
        "jacobian": asdict(jacobian_config), "factor": asdict(factor_config),
        "product_output": asdict(output_config),
        "heaps": {"innovation_covariance": [[SIGMA**2]], "mapping": "Linear coefficients used as free U entries; final mean chosen to preserve the original intercept."},
        "noise_policies": noise_policies.to_dict(orient="records"),
        "validation_seed": VALIDATION_SEED, "validation_histories": 32,
        "history_shape": [1, 12], "history_distribution": "Uniform(-1, 1)",
        "formula_absolute_tolerance": 1e-12, "scale_relative_tolerance": 1e-13,
        "trajectory_simulation": False,
    },
    "provenance": {
        "executed_at_utc": datetime.now(timezone.utc).isoformat(), "git_revision": git_revision,
        "source_worktree_status": git_status, "python": sys.version, "platform": platform.platform(),
        "packages": {name: importlib.metadata.version(name) for name in ["numpy", "pandas", "nbformat", "nbclient"]},
        "sha256": {path: hashlib.sha256((ROOT/path).read_bytes()).hexdigest() for path in source_paths},
    },
    "metrics": validation.to_dict(orient="records"),
    "formulas": comparison.reset_index().to_dict(orient="records"),
    "summary": {"sampled_asts": 9, "ast_transformations": 27, "heaps_constructions": 5,
                "pointwise_checks": int(validation["Históricos testados"].sum()),
                "max_absolute_error": float(validation["Máximo erro absoluto"].max()),
                "all_passed": bool(validation["Aprovado"].all())},
    "scientific_summary": "Os resultados mostram os efeitos distintos das construções sobre a equação. O modo produto certifica limite de segundo momento, e o modo fatores certifica contração em momento. Heaps usa uma convenção explícita de parâmetros livres para os casos lineares. As verificações numéricas conferem a implementação; as garantias vêm das condições matemáticas verificadas pelos construtores.",
    "reproduction": "python -m jupyter nbconvert --to notebook --execute --inplace --ExecutePreprocessor.timeout=300 experiments/abstract_syntax_tree/8_generation_overview.ipynb",
    "references": ["shao2007spectral", "heaps2023stationarity", "tran2019convergence", "laraba2016setinvariance"],
}
display(JSON(record))

<IPython.core.display.JSON object>

**O que foi feito:** O registro completo ficou no próprio notebook: configurações, sementes, hashes dos arquivos, versões, fórmulas, métricas, interpretação e comando de reprodução. Os arquivos auxiliares de configuração e métricas são dispensados por esse registro interno.